# Smoke tests de OP-07 `read_trips`

Este notebook prueba la operación pública `read_trips()` a nivel smoke test.

Objetivo:
- verificar que `read_trips()` reconstruye un `TripDataset` desde un artefacto formal de trips;
- verificar lectura desde sidecar `trips.metadata.json`;
- cubrir backend Parquet y backend Feather;
- verificar la regla post-read `metadata["is_validated"] = False`;
- verificar evento `read_trips` cuando `keep_metadata=True`;
- probar fallas públicas relevantes de lectura;
- incluir algunos casos `write_trips + read_trips` solo como preparación realista del artefacto.

Convenciones:
- los tests usan `assert`;
- los artefactos se escriben en una carpeta visible junto al notebook;
- este notebook cubre solo smoke tests de OP-07;
- OP-06 aparece solo como setup en algunos casos, no como subject principal del test.

## Bloque 0. Preparación

### 0.1 Imports generales

Qué prepara: imports básicos, JSON, filesystem, pandas y PyArrow para inspeccionar los artefactos persistidos.

In [1]:
import copy
import json
import shutil
from pathlib import Path

import pandas as pd

import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.feather as feather

### 0.2 Imports del módulo

Qué prepara: clases mínimas para construir un `TripDataset` sintético, escribir artefactos de setup y probar la operación pública `read_trips`.

In [2]:
from pylondrina.schema import (
    DomainSpec,
    FieldSpec,
    TripSchema,
    TripSchemaEffective,
)

from pylondrina.datasets import TripDataset
from pylondrina.errors import ExportError

from pylondrina.io.trips import (
    write_trips,
    read_trips,
    WriteTripsOptions,
    ReadTripsOptions,
)

### 0.3 Helpers de apoyo para test

Qué prepara: utilidades pequeñas para asserts, mensajes, lectura de sidecar e inspección de issues.

In [3]:
def show_ok(label: str):
    print(f"OK - {label}")

def get_issue_codes(issues):
    return [i.code if hasattr(i, "code") else i.get("code") for i in issues]


def assert_issue_present(issues, code: str):
    codes = get_issue_codes(issues)
    assert code in codes, f"No se encontró el issue {code}. Codes actuales: {codes}"


def assert_issue_absent(issues, code: str):
    codes = get_issue_codes(issues)
    assert code not in codes, f"Se encontró inesperadamente el issue {code}. Codes actuales: {codes}"


def load_sidecar(artifact_dir: Path) -> dict:
    sidecar_path = artifact_dir / "trips.metadata.json"
    assert sidecar_path.exists(), f"No existe sidecar: {sidecar_path}"
    return json.loads(sidecar_path.read_text(encoding="utf-8"))


def write_sidecar(artifact_dir: Path, payload: dict) -> None:
    sidecar_path = artifact_dir / "trips.metadata.json"
    sidecar_path.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


def file_size(path: Path) -> int:
    assert path.exists(), f"No existe archivo: {path}"
    return path.stat().st_size


def assert_loaded_data_equivalent(left: pd.DataFrame, right: pd.DataFrame):
    pd.testing.assert_frame_equal(
        left.reset_index(drop=True),
        right.reset_index(drop=True),
        check_dtype=False,
        check_categorical=False,
    )

### 0.4 Configuración visual

In [4]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 120)

print("Imports OK")
show_ok("Sección 0 cargada")

Imports OK
OK - Sección 0 cargada


## Bloque 1. Fixtures reutilizables mínimas

Qué prepara: factories pequeñas para crear un `TripDataset` sintético validado y artefactos formales mínimos.

In [5]:
def make_field(
    name: str,
    dtype: str,
    *,
    required: bool = False,
    constraints: dict | None = None,
    domain: DomainSpec | None = None,
) -> FieldSpec:
    return FieldSpec(
        name=name,
        dtype=dtype,
        required=required,
        constraints=constraints,
        domain=domain,
    )


def make_trip_schema(fields: list[FieldSpec], *, version: str = "1.1") -> TripSchema:
    return TripSchema(
        version=version,
        fields={f.name: f for f in fields},
        required=[f.name for f in fields if f.required],
        semantic_rules=None,
    )


def make_trip_schema_effective(
    *,
    dtype_effective: dict | None = None,
    overrides: dict | None = None,
    domains_effective: dict | None = None,
    temporal: dict | None = None,
    fields_effective: list | None = None,
) -> TripSchemaEffective:
    return TripSchemaEffective(
        dtype_effective=dtype_effective or {},
        overrides=overrides or {},
        domains_effective=domains_effective or {},
        temporal=temporal or {},
        fields_effective=fields_effective or [],
    )


def make_trip_df() -> pd.DataFrame:
    return pd.DataFrame(
        {
            "movement_id": ["m1", "m2", "m3"],
            "trip_id": ["t1", "t2", "t3"],
            "movement_seq": [0, 0, 0],
            "user_id": ["u1", "u2", "u3"],
            "origin_latitude": [-33.45, -33.46, -33.47],
            "origin_longitude": [-70.66, -70.67, -70.68],
            "destination_latitude": [-33.41, -33.42, -33.43],
            "destination_longitude": [-70.61, -70.62, -70.63],
            "mode": ["bus", "metro", "bus"],
            "purpose": ["work", "study", "work"],
            "comment": ["a", "b", "c"],
            "trip_weight": [1.0, 2.5, 1.2],
        }
    )


def make_trip_schema_minimal(*, version: str = "1.1") -> TripSchema:
    return make_trip_schema(
        [
            make_field("movement_id", "string", required=True),
            make_field("trip_id", "string", required=True),
            make_field("movement_seq", "int", required=True),
            make_field("user_id", "string", required=True),
            make_field("origin_latitude", "float", required=True),
            make_field("origin_longitude", "float", required=True),
            make_field("destination_latitude", "float", required=True),
            make_field("destination_longitude", "float", required=True),
            make_field(
                "mode",
                "categorical",
                required=False,
                domain=DomainSpec(values=["bus", "metro", "walk", "car"], extendable=True),
            ),
            make_field(
                "purpose",
                "categorical",
                required=False,
                domain=DomainSpec(values=["work", "study", "health"], extendable=True),
            ),
            make_field("comment", "string", required=False),
            make_field("trip_weight", "float", required=False),
        ],
        version=version,
    )


def make_trip_schema_effective_minimal() -> TripSchemaEffective:
    return make_trip_schema_effective(
        dtype_effective={
            "mode": "categorical",
            "purpose": "categorical",
            "trip_weight": "float",
        },
        domains_effective={
            "mode": {"values": ["bus", "metro", "walk", "car"]},
            "purpose": {"values": ["work", "study", "health"]},
        },
        temporal={"tier": "tier_3"},
        fields_effective=[
            "movement_id",
            "trip_id",
            "movement_seq",
            "user_id",
            "origin_latitude",
            "origin_longitude",
            "destination_latitude",
            "destination_longitude",
            "mode",
            "purpose",
            "comment",
            "trip_weight",
        ],
    )


def make_tripdataset(
    *,
    validated: bool = True,
    include_dataset_id: bool = True,
    include_artifact_id: bool = False,
) -> TripDataset:
    schema = make_trip_schema_minimal()
    schema_effective = make_trip_schema_effective_minimal()

    metadata = {
        "is_validated": validated,
        "events": [],
        "mappings": {
            "field_correspondence": {
                "movement_id": "movement_id_src",
                "mode": "mode_src",
            },
            "value_correspondence": {
                "mode": {
                    "micro": "bus",
                    "subte": "metro",
                }
            },
        },
        "domains_effective": copy.deepcopy(schema_effective.domains_effective),
        "temporal": {"tier": "tier_3"},
    }

    if include_dataset_id:
        metadata["dataset_id"] = "dset_test_001"
    if include_artifact_id:
        metadata["artifact_id"] = "art_test_001"

    provenance = {
        "source": {"name": "synthetic", "entity": "trips"},
        "ingestion": {"created_at_utc": "2026-04-04T00:00:00Z"},
    }

    return TripDataset(
        data=make_trip_df(),
        schema=schema,
        schema_version=schema.version,
        provenance=provenance,
        field_correspondence={"movement_id": "movement_id_src", "mode": "mode_src"},
        value_correspondence={"mode": {"micro": "bus", "subte": "metro"}},
        metadata=metadata,
        schema_effective=schema_effective,
    )

## Bloque 2. Carpeta visible para artefactos de smoke tests

Qué prepara: una carpeta local, junto al notebook, para inspeccionar manualmente los bundles leídos por `read_trips`.

La carpeta se reinicia al ejecutar este bloque.

In [6]:
SMOKE_ROOT = Path("./tmp_op07_read_trips_smoke")

def reset_smoke_root() -> Path:
    if SMOKE_ROOT.exists():
        shutil.rmtree(SMOKE_ROOT)
    SMOKE_ROOT.mkdir(parents=True, exist_ok=True)
    return SMOKE_ROOT


def make_case_dir(case_name: str) -> Path:
    case_dir = SMOKE_ROOT / case_name
    case_dir.mkdir(parents=True, exist_ok=True)
    return case_dir


root = reset_smoke_root()
print("SMOKE_ROOT =", root.resolve())
show_ok("Bloque 2 - carpeta visible preparada")

SMOKE_ROOT = C:\projects\pylondrina\notebooks\testing\io_trips\tmp_op07_read_trips_smoke
OK - Bloque 2 - carpeta visible preparada


## Bloque 3. Smoke tests de `read_trips`

### Test 3.1 - smoke test de `read_trips` happy path usando snapshot de schema

Qué prueba: lectura formal exitosa desde un bundle Parquet generado previamente.

La escritura con `write_trips()` se usa solo para preparar un artefacto realista. El foco del test es que `read_trips()`:

- use el schema persistido en el sidecar;
- reconstruya un `TripDataset`;
- retorne `OperationReport.ok=True`;
- fuerce `metadata["is_validated"] = False`;
- agregue evento `read_trips` cuando `keep_metadata=True`;
- preserve `dataset_id` y `artifact_id`;
- cargue los datos de forma equivalente.

In [7]:
case_dir = make_case_dir("case_01_read_from_snapshot_parquet")
artifact_dir = case_dir / "artifact"

trips = make_tripdataset(validated=True)
data_before_write = trips.data.copy(deep=True)

write_report = write_trips(
    trips,
    artifact_dir,
    options=WriteTripsOptions(
        mode="error_if_exists",
        require_validated=True,
        storage_format="parquet",
        parquet_compression="snappy",
        normalize_artifact_dir=False,
    ),
)

loaded, read_report = read_trips(
    artifact_dir,
    options=ReadTripsOptions(
        schema=None,
        strict=False,
        keep_metadata=True,
    ),
)

assert write_report.ok is True
assert read_report.ok is True

assert isinstance(loaded, TripDataset)
assert loaded.schema.version == trips.schema.version
assert loaded.schema.to_dict() == trips.schema.to_dict()
assert loaded.schema_effective.to_dict() == trips.schema_effective.to_dict()

assert read_report.parameters["schema"]["source"] == "metadata"
assert read_report.summary["schema_source"] == "metadata"
assert read_report.summary["storage_format"] == "parquet"
assert read_report.summary["n_rows"] == len(trips.data)
assert read_report.summary["n_columns"] == len(trips.data.columns)

# Post-read: lectura formal no equivale a certificación.
assert loaded.metadata["is_validated"] is False
assert_issue_present(read_report.issues, "READ.METADATA.VALIDATED_FORCED_FALSE")

# Identidad lógica y de artefacto preservada.
assert loaded.metadata["dataset_id"] == trips.metadata["dataset_id"]
assert loaded.metadata["artifact_id"] == trips.metadata["artifact_id"]
assert read_report.summary["dataset_id"] == trips.metadata["dataset_id"]
assert read_report.summary["artifact_id"] == trips.metadata["artifact_id"]

# Evento de lectura.
assert loaded.metadata["events"][-1]["op"] == "read_trips"
assert loaded.metadata["events"][-1]["summary"] == read_report.summary

# Datos equivalentes.
assert_loaded_data_equivalent(loaded.data, data_before_write)

display(read_report)
show_ok("Test 3.1 - read_trips happy path desde snapshot Parquet")

OperationReport(ok=True, issues=[Issue(level='info', code='READ.METADATA.VALIDATED_FORCED_FALSE', message="Se forzó metadata['is_validated']=False tras la lectura formal del artefacto de trips.", field=None, source_field=None, row_count=None, details={'previous_value': True, 'new_value': False, 'action': 'force_unvalidated'})], summary={'n_rows': 3, 'n_columns': 12, 'path': 'tmp_op07_read_trips_smoke\\case_01_read_from_snapshot_parquet\\artifact', 'storage_format': 'parquet', 'schema_source': 'metadata', 'schema_mismatch': False, 'dataset_id': 'dset_test_001', 'dataset_id_status': 'loaded', 'artifact_id': 'art_fdc266dd-0a1d-442d-9753-17c6e90d902d', 'artifact_id_status': 'loaded'}, parameters={'path': 'tmp_op07_read_trips_smoke\\case_01_read_from_snapshot_parquet\\artifact', 'strict': False, 'keep_metadata': True, 'schema': {'source': 'metadata', 'version': '1.1'}})

OK - Test 3.1 - read_trips happy path desde snapshot Parquet


### Test 3.2 - smoke test de `read_trips` happy path con Feather

Qué prueba: lectura formal exitosa desde un bundle Feather. Este caso es necesario porque la implementación vigente de OP-07 ya no es Parquet-only.

Verifica:
- resolución de `storage.format = "feather"`;
- lectura de `trips.feather`;
- coherencia de summary y parameters;
- regla `is_validated=False` post-read.

In [8]:
case_dir = make_case_dir("case_02_read_from_snapshot_feather")
artifact_dir = case_dir / "artifact"

trips = make_tripdataset(validated=True)
data_before_write = trips.data.copy(deep=True)

write_report = write_trips(
    trips,
    artifact_dir,
    options=WriteTripsOptions(
        mode="error_if_exists",
        require_validated=True,
        storage_format="feather",
        feather_compression="lz4",
        normalize_artifact_dir=False,
    ),
)

loaded, read_report = read_trips(
    artifact_dir,
    options=ReadTripsOptions(
        schema=None,
        strict=False,
        keep_metadata=True,
    ),
)

assert write_report.ok is True
assert read_report.ok is True

assert (artifact_dir / "trips.feather").exists()
assert not (artifact_dir / "trips.parquet").exists()

assert read_report.parameters["schema"]["source"] == "metadata"
assert read_report.summary["schema_source"] == "metadata"
assert read_report.summary["storage_format"] == "feather"

assert loaded.schema.version == trips.schema.version
assert loaded.schema_effective.to_dict() == trips.schema_effective.to_dict()

assert loaded.metadata["is_validated"] is False
assert_issue_present(read_report.issues, "READ.METADATA.VALIDATED_FORCED_FALSE")

assert loaded.metadata["dataset_id"] == trips.metadata["dataset_id"]
assert loaded.metadata["artifact_id"] == trips.metadata["artifact_id"]
assert loaded.metadata["events"][-1]["op"] == "read_trips"

assert_loaded_data_equivalent(loaded.data, data_before_write)

sidecar = load_sidecar(artifact_dir)
assert sidecar["storage"]["format"] == "feather"
assert sidecar["files"]["data"] == "trips.feather"

display(read_report.summary)
show_ok("Test 3.2 - read_trips happy path Feather")

{'n_rows': 3,
 'n_columns': 12,
 'path': 'tmp_op07_read_trips_smoke\\case_02_read_from_snapshot_feather\\artifact',
 'storage_format': 'feather',
 'schema_source': 'metadata',
 'schema_mismatch': False,
 'dataset_id': 'dset_test_001',
 'dataset_id_status': 'loaded',
 'artifact_id': 'art_993a952e-5d1d-470c-a54b-5f9ae5a0a26a',
 'artifact_id_status': 'loaded'}

OK - Test 3.2 - read_trips happy path Feather


### Test 3.3 - smoke test de `read_trips` con schema explícito

Qué prueba: precedencia de `ReadTripsOptions.schema` sobre el snapshot persistido.

El schema entregado por el usuario se usa como schema vivo del dataset reconstruido. Como la versión difiere del sidecar, se espera un warning `READ.SCHEMA.MISMATCH`, pero la lectura sigue siendo recuperable con `strict=False`.

In [9]:
case_dir = make_case_dir("case_03_read_with_schema_option")
artifact_dir = case_dir / "artifact"

trips = make_tripdataset(validated=True)

write_report = write_trips(
    trips,
    artifact_dir,
    options=WriteTripsOptions(
        mode="error_if_exists",
        require_validated=True,
        storage_format="parquet",
        parquet_compression="snappy",
        normalize_artifact_dir=False,
    ),
)

schema_override = make_trip_schema_minimal(version="1.1-override")

loaded, read_report = read_trips(
    artifact_dir,
    options=ReadTripsOptions(
        schema=schema_override,
        strict=False,
        keep_metadata=True,
    ),
)

assert write_report.ok is True
assert read_report.ok is True

assert loaded.schema.version == "1.1-override"
assert read_report.parameters["schema"]["source"] == "options"
assert read_report.parameters["schema"]["version"] == "1.1-override"
assert read_report.summary["schema_source"] == "options"
assert read_report.summary["schema_mismatch"] is True

assert_issue_present(read_report.issues, "READ.SCHEMA.MISMATCH")
assert_issue_present(read_report.issues, "READ.METADATA.VALIDATED_FORCED_FALSE")

assert loaded.metadata["is_validated"] is False
assert loaded.metadata["events"][-1]["op"] == "read_trips"

display(read_report.issues)
display(read_report.summary)
show_ok("Test 3.3 - read_trips con schema explícito")

[Issue(level='warning', code='READ.SCHEMA.MISMATCH', message='El schema provisto por options no coincide con el snapshot persistido en el sidecar; se usará options.schema según precedencia.', field=None, source_field=None, row_count=None, details={'schema_source': 'options', 'schema_mismatch': True, 'version_options': '1.1-override', 'version_metadata': '1.1', 'required_diff': [], 'fields_diff_sample': [], 'fields_diff_total': 0, 'action': 'use_options_schema'}),
 Issue(level='info', code='READ.METADATA.VALIDATED_FORCED_FALSE', message="Se forzó metadata['is_validated']=False tras la lectura formal del artefacto de trips.", field=None, source_field=None, row_count=None, details={'previous_value': True, 'new_value': False, 'action': 'force_unvalidated'})]

{'n_rows': 3,
 'n_columns': 12,
 'path': 'tmp_op07_read_trips_smoke\\case_03_read_with_schema_option\\artifact',
 'storage_format': 'parquet',
 'schema_source': 'options',
 'schema_mismatch': True,
 'dataset_id': 'dset_test_001',
 'dataset_id_status': 'loaded',
 'artifact_id': 'art_d50ec1bd-f620-41a0-9939-b9cec6ef2b28',
 'artifact_id_status': 'loaded'}

OK - Test 3.3 - read_trips con schema explícito


### Test 3.4 - smoke test de `read_trips` con `keep_metadata=False`

Qué prueba: cuando `keep_metadata=False`, `read_trips()` reconstruye el dataset y fuerza `is_validated=False`, pero no agrega evento `read_trips` a `metadata["events"]`.

Esto separa la lectura formal de la política de trazabilidad en metadata.

In [10]:
case_dir = make_case_dir("case_04_read_keep_metadata_false")
artifact_dir = case_dir / "artifact"

trips = make_tripdataset(validated=True)

write_report = write_trips(
    trips,
    artifact_dir,
    options=WriteTripsOptions(
        mode="error_if_exists",
        require_validated=True,
        storage_format="parquet",
        parquet_compression="snappy",
        normalize_artifact_dir=False,
    ),
)

sidecar = load_sidecar(artifact_dir)
events_before_read = copy.deepcopy(sidecar["metadata"]["events"])

loaded, read_report = read_trips(
    artifact_dir,
    options=ReadTripsOptions(
        schema=None,
        strict=False,
        keep_metadata=False,
    ),
)

assert write_report.ok is True
assert read_report.ok is True

assert loaded.metadata["is_validated"] is False
assert_issue_present(read_report.issues, "READ.METADATA.VALIDATED_FORCED_FALSE")

# No debe agregarse evento read_trips.
assert loaded.metadata["events"] == events_before_read
assert all(ev["op"] != "read_trips" for ev in loaded.metadata["events"])

# El reporte sí existe y sí trae parámetros/summary.
assert read_report.parameters["keep_metadata"] is False
assert read_report.summary["storage_format"] == "parquet"

display(read_report.summary)
show_ok("Test 3.4 - read_trips keep_metadata=False")

{'n_rows': 3,
 'n_columns': 12,
 'path': 'tmp_op07_read_trips_smoke\\case_04_read_keep_metadata_false\\artifact',
 'storage_format': 'parquet',
 'schema_source': 'metadata',
 'schema_mismatch': False,
 'dataset_id': 'dset_test_001',
 'dataset_id_status': 'loaded',
 'artifact_id': 'art_2b400dae-2674-455b-a48c-4b11f91f07ed',
 'artifact_id_status': 'loaded'}

OK - Test 3.4 - read_trips keep_metadata=False


### Test 3.5 - smoke test de resolución automática del sufijo `.golondrina`

Qué prueba: si el path exacto no existe y no termina en `.golondrina`, `read_trips()` intenta leer automáticamente `path.golondrina`.

En este caso `write_trips()` materializa el bundle con normalización activa, y `read_trips()` recibe el path base sin sufijo.

In [11]:
case_dir = make_case_dir("case_05_read_auto_suffix")
base_path = case_dir / "artifact_auto_suffix"
expected_artifact_dir = case_dir / "artifact_auto_suffix.golondrina"

trips = make_tripdataset(validated=True)

write_report = write_trips(
    trips,
    base_path,
    options=WriteTripsOptions(
        mode="error_if_exists",
        require_validated=True,
        storage_format="parquet",
        parquet_compression="snappy",
        normalize_artifact_dir=True,
    ),
)

assert write_report.ok is True
assert expected_artifact_dir.exists()
assert not base_path.exists()

loaded, read_report = read_trips(
    base_path,
    options=ReadTripsOptions(
        schema=None,
        strict=False,
        keep_metadata=True,
    ),
)

assert read_report.ok is True
assert Path(read_report.summary["path"]) == expected_artifact_dir
assert read_report.summary["storage_format"] == "parquet"
assert loaded.metadata["dataset_id"] == trips.metadata["dataset_id"]
assert loaded.metadata["artifact_id"] == trips.metadata["artifact_id"]
assert loaded.metadata["events"][-1]["op"] == "read_trips"

display(read_report.summary)
show_ok("Test 3.5 - read_trips resolución automática .golondrina")

{'n_rows': 3,
 'n_columns': 12,
 'path': 'tmp_op07_read_trips_smoke\\case_05_read_auto_suffix\\artifact_auto_suffix.golondrina',
 'storage_format': 'parquet',
 'schema_source': 'metadata',
 'schema_mismatch': False,
 'dataset_id': 'dset_test_001',
 'dataset_id_status': 'loaded',
 'artifact_id': 'art_1dbb73ef-5a76-4121-91c3-e6664e8f2def',
 'artifact_id_status': 'loaded'}

OK - Test 3.5 - read_trips resolución automática .golondrina


### Test 3.6 - smoke test fatal por sidecar faltante

Qué prueba: `read_trips()` no interpreta un archivo tabular suelto como artefacto formal. Si falta `trips.metadata.json`, la lectura debe abortar con `ExportError`.

In [12]:
case_dir = make_case_dir("case_06_read_fatal_missing_sidecar")
artifact_dir = case_dir / "artifact"
artifact_dir.mkdir(parents=True, exist_ok=True)

make_trip_df().to_parquet(
    artifact_dir / "trips.parquet",
    index=False,
    compression="snappy",
    engine="pyarrow",
)

raised = None
try:
    read_trips(
        artifact_dir,
        options=ReadTripsOptions(
            schema=None,
            strict=False,
            keep_metadata=True,
        ),
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, ExportError)
assert getattr(raised, "code", None) == "READ.LAYOUT.MISSING_SIDECAR"

assert not (artifact_dir / "trips.metadata.json").exists()

display(raised)
show_ok("Test 3.6 - fatal de read_trips por sidecar faltante")

ExportError(message="El artefacto formal de trips no contiene el sidecar obligatorio 'trips.metadata.json'.", code='READ.LAYOUT.MISSING_SIDECAR', details={'path': 'tmp_op07_read_trips_smoke\\case_06_read_fatal_missing_sidecar\\artifact', 'resolved_path': 'tmp_op07_read_trips_smoke\\case_06_read_fatal_missing_sidecar\\artifact', 'expected_file': 'trips.metadata.json', 'files_present_sample': ['trips.parquet'], 'files_present_total': 1, 'action': 'abort'}, issue=Issue(level='error', code='READ.LAYOUT.MISSING_SIDECAR', message="El artefacto formal de trips no contiene el sidecar obligatorio 'trips.metadata.json'.", field=None, source_field=None, row_count=None, details={'path': 'tmp_op07_read_trips_smoke\\case_06_read_fatal_missing_sidecar\\artifact', 'resolved_path': 'tmp_op07_read_trips_smoke\\case_06_read_fatal_missing_sidecar\\artifact', 'expected_file': 'trips.metadata.json', 'files_present_sample': ['trips.parquet'], 'files_present_total': 1, 'action': 'abort'}), issues=(Issue(level

OK - Test 3.6 - fatal de read_trips por sidecar faltante


### Test 3.7 - smoke test fatal por sidecar legacy

Qué prueba: `read_trips()` debe rechazar `metadata.json` legacy cuando falta el sidecar formal `trips.metadata.json`.

Esto evita confundir artefactos antiguos o informales con persistencia formal de trips.

In [13]:
case_dir = make_case_dir("case_07_read_fatal_legacy_sidecar")
artifact_dir = case_dir / "artifact"
artifact_dir.mkdir(parents=True, exist_ok=True)

make_trip_df().to_parquet(
    artifact_dir / "trips.parquet",
    index=False,
    compression="snappy",
    engine="pyarrow",
)

(artifact_dir / "metadata.json").write_text(
    json.dumps({"legacy": True}, ensure_ascii=False),
    encoding="utf-8",
)

raised = None
try:
    read_trips(
        artifact_dir,
        options=ReadTripsOptions(
            schema=None,
            strict=False,
            keep_metadata=True,
        ),
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, ExportError)
assert getattr(raised, "code", None) == "READ.LAYOUT.LEGACY_SIDECAR_DETECTED"

display(raised)
show_ok("Test 3.7 - fatal de read_trips por sidecar legacy")

ExportError(message="Se detectó un sidecar legacy 'metadata.json', pero falta el sidecar formal 'trips.metadata.json'; el artefacto no es válido para read_trips v1.1.", code='READ.LAYOUT.LEGACY_SIDECAR_DETECTED', details={'path': 'tmp_op07_read_trips_smoke\\case_07_read_fatal_legacy_sidecar\\artifact', 'resolved_path': 'tmp_op07_read_trips_smoke\\case_07_read_fatal_legacy_sidecar\\artifact', 'legacy_file': 'metadata.json', 'expected_file': 'trips.metadata.json', 'action': 'abort'}, issue=Issue(level='error', code='READ.LAYOUT.LEGACY_SIDECAR_DETECTED', message="Se detectó un sidecar legacy 'metadata.json', pero falta el sidecar formal 'trips.metadata.json'; el artefacto no es válido para read_trips v1.1.", field=None, source_field=None, row_count=None, details={'path': 'tmp_op07_read_trips_smoke\\case_07_read_fatal_legacy_sidecar\\artifact', 'resolved_path': 'tmp_op07_read_trips_smoke\\case_07_read_fatal_legacy_sidecar\\artifact', 'legacy_file': 'metadata.json', 'expected_file': 'trips.

OK - Test 3.7 - fatal de read_trips por sidecar legacy


### Test 3.8 - smoke test degradado con recovery `strict=False`

Qué prueba: un sidecar parcialmente degradado puede reconstruirse en modo no estricto.

Se degrada manualmente un artefacto correcto:
- se elimina `schema_effective`;
- se deja `artifact_id=None`.

Con `strict=False`, la lectura debe:
- retornar `OperationReport.ok=True`;
- defaultar `schema_effective`;
- dejar `artifact_id=None`;
- forzar `is_validated=False`;
- registrar issues de recuperación.

In [14]:
case_dir = make_case_dir("case_08_read_degraded_recovery")
artifact_dir = case_dir / "artifact"

trips = make_tripdataset(validated=True)

write_report = write_trips(
    trips,
    artifact_dir,
    options=WriteTripsOptions(
        mode="error_if_exists",
        require_validated=True,
        storage_format="parquet",
        parquet_compression="snappy",
        normalize_artifact_dir=False,
    ),
)

assert write_report.ok is True

payload_bad = load_sidecar(artifact_dir)
payload_bad.pop("schema_effective", None)
payload_bad["artifact_id"] = None
payload_bad["metadata"]["artifact_id"] = None

write_sidecar(artifact_dir, payload_bad)

loaded, read_report = read_trips(
    artifact_dir,
    options=ReadTripsOptions(
        schema=None,
        strict=False,
        keep_metadata=True,
    ),
)

assert read_report.ok is True

assert_issue_present(read_report.issues, "READ.SCHEMA_EFFECTIVE.DEFAULTED")
assert_issue_present(read_report.issues, "READ.METADATA.ARTIFACT_ID_SET_NONE")
assert_issue_present(read_report.issues, "READ.METADATA.VALIDATED_FORCED_FALSE")

assert loaded.metadata["is_validated"] is False
assert loaded.metadata["artifact_id"] is None
assert loaded.schema_effective.to_dict() == TripSchemaEffective().to_dict()
assert loaded.metadata["events"][-1]["op"] == "read_trips"

display(read_report.issues)
display(read_report.summary)
show_ok("Test 3.8 - read_trips degradado con recovery strict=False")

[Issue(level='warning', code='READ.SCHEMA_EFFECTIVE.DEFAULTED', message='schema_effective no está disponible o no es interpretable; se reconstruirá un estado efectivo vacío/default.', field=None, source_field=None, row_count=None, details={'reason': 'missing_schema_effective_snapshot', 'strict': False, 'action': 'default_empty_schema_effective'}),
 Issue(level='warning', code='READ.METADATA.ARTIFACT_ID_SET_NONE', message='El artifact_id persistido faltaba o era inválido; se dejará artifact_id=None en el dataset cargado.', field=None, source_field=None, row_count=None, details={'artifact_id': None, 'artifact_id_status': 'missing_or_invalid', 'previous_value': None, 'reason': 'missing_or_invalid_in_sidecar', 'action': 'set_none'}),
 Issue(level='info', code='READ.METADATA.VALIDATED_FORCED_FALSE', message="Se forzó metadata['is_validated']=False tras la lectura formal del artefacto de trips.", field=None, source_field=None, row_count=None, details={'previous_value': True, 'new_value': Fal

{'n_rows': 3,
 'n_columns': 12,
 'path': 'tmp_op07_read_trips_smoke\\case_08_read_degraded_recovery\\artifact',
 'storage_format': 'parquet',
 'schema_source': 'metadata',
 'schema_mismatch': False,
 'dataset_id': 'dset_test_001',
 'dataset_id_status': 'loaded',
 'artifact_id': None,
 'artifact_id_status': 'missing_or_invalid'}

OK - Test 3.8 - read_trips degradado con recovery strict=False


### Test 3.9 - smoke test fatal degradado con `strict=True`

Qué prueba: el mismo artefacto degradado del caso anterior debe abortar cuando `strict=True`.

En particular, la ausencia de `schema_effective` no debe recuperarse silenciosamente en modo estricto.

In [15]:
case_dir = make_case_dir("case_09_read_degraded_strict_true")
artifact_dir = case_dir / "artifact"

trips = make_tripdataset(validated=True)

write_report = write_trips(
    trips,
    artifact_dir,
    options=WriteTripsOptions(
        mode="error_if_exists",
        require_validated=True,
        storage_format="parquet",
        parquet_compression="snappy",
        normalize_artifact_dir=False,
    ),
)

assert write_report.ok is True

payload_bad = load_sidecar(artifact_dir)
payload_bad.pop("schema_effective", None)

write_sidecar(artifact_dir, payload_bad)

raised = None
try:
    read_trips(
        artifact_dir,
        options=ReadTripsOptions(
            schema=None,
            strict=True,
            keep_metadata=True,
        ),
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, ExportError)
assert getattr(raised, "code", None) == "READ.SCHEMA_EFFECTIVE.DEFAULTED"

display(raised)
show_ok("Test 3.9 - read_trips degradado fatal con strict=True")

ExportError(message='schema_effective no está disponible o no es interpretable; se reconstruirá un estado efectivo vacío/default.', code='READ.SCHEMA_EFFECTIVE.DEFAULTED', details={'reason': 'schema_effective unavailable or invalid in strict mode', 'strict': True, 'action': 'default_empty_schema_effective'}, issue=Issue(level='warning', code='READ.SCHEMA_EFFECTIVE.DEFAULTED', message='schema_effective no está disponible o no es interpretable; se reconstruirá un estado efectivo vacío/default.', field=None, source_field=None, row_count=None, details={'reason': 'schema_effective unavailable or invalid in strict mode', 'strict': True, 'action': 'default_empty_schema_effective'}), issues=(Issue(level='warning', code='READ.SCHEMA_EFFECTIVE.DEFAULTED', message='schema_effective no está disponible o no es interpretable; se reconstruirá un estado efectivo vacío/default.', field=None, source_field=None, row_count=None, details={'reason': 'schema_effective unavailable or invalid in strict mode', 

OK - Test 3.9 - read_trips degradado fatal con strict=True


### Test 3.10 - smoke test fatal por mismatch backend/data file

Qué prueba: el sidecar es la fuente de verdad del backend. Si `storage.format="feather"` pero `files.data` apunta a `trips.parquet`, el artefacto es incoherente y `read_trips()` debe abortar.

In [16]:
case_dir = make_case_dir("case_10_read_fatal_backend_file_mismatch")
artifact_dir = case_dir / "artifact"

trips = make_tripdataset(validated=True)

write_report = write_trips(
    trips,
    artifact_dir,
    options=WriteTripsOptions(
        mode="error_if_exists",
        require_validated=True,
        storage_format="feather",
        feather_compression="lz4",
        normalize_artifact_dir=False,
    ),
)

assert write_report.ok is True
assert (artifact_dir / "trips.feather").exists()

payload_bad = load_sidecar(artifact_dir)
payload_bad["files"]["data"] = "trips.parquet"

write_sidecar(artifact_dir, payload_bad)

raised = None
try:
    read_trips(
        artifact_dir,
        options=ReadTripsOptions(
            schema=None,
            strict=False,
            keep_metadata=True,
        ),
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, ExportError)
assert getattr(raised, "code", None) == "READ.LAYOUT.DATA_FILE_MISMATCH"

display(raised)
show_ok("Test 3.10 - fatal por mismatch storage.format/files.data")

ExportError(message="El sidecar declara un archivo de datos 'trips.parquet' inconsistente con storage.format='feather'; se esperaba 'trips.feather'.", code='READ.LAYOUT.DATA_FILE_MISMATCH', details={'expected_file': 'trips.feather', 'declared_file': 'trips.parquet', 'storage_format': 'feather', 'action': 'abort'}, issue=Issue(level='error', code='READ.LAYOUT.DATA_FILE_MISMATCH', message="El sidecar declara un archivo de datos 'trips.parquet' inconsistente con storage.format='feather'; se esperaba 'trips.feather'.", field=None, source_field=None, row_count=None, details={'expected_file': 'trips.feather', 'declared_file': 'trips.parquet', 'storage_format': 'feather', 'action': 'abort'}), issues=(Issue(level='error', code='READ.LAYOUT.DATA_FILE_MISMATCH', message="El sidecar declara un archivo de datos 'trips.parquet' inconsistente con storage.format='feather'; se esperaba 'trips.feather'.", field=None, source_field=None, row_count=None, details={'expected_file': 'trips.feather', 'declare

OK - Test 3.10 - fatal por mismatch storage.format/files.data


### Test 3.11 - smoke test integrado de round-trip mínimo

Qué prueba: caso mínimo `write_trips + read_trips` para asegurar que el artefacto escrito por OP-06 puede ser reconstruido por OP-07.

El foco sigue en la lectura:
- datos equivalentes;
- schema y schema_effective equivalentes;
- identidad preservada;
- `write_trips` persistido en metadata;
- `read_trips` agregado al final;
- `is_validated=False` tras lectura.

In [17]:
case_dir = make_case_dir("case_11_roundtrip_minimal")
artifact_dir = case_dir / "artifact"

trips_original = make_tripdataset(validated=True)
data_original = trips_original.data.copy(deep=True)
schema_original = copy.deepcopy(trips_original.schema)
schema_effective_original = copy.deepcopy(trips_original.schema_effective)

write_report = write_trips(
    trips_original,
    artifact_dir,
    options=WriteTripsOptions(
        mode="error_if_exists",
        require_validated=True,
        storage_format="parquet",
        parquet_compression="snappy",
        normalize_artifact_dir=False,
    ),
)

loaded, read_report = read_trips(
    artifact_dir,
    options=ReadTripsOptions(
        schema=None,
        strict=False,
        keep_metadata=True,
    ),
)

assert write_report.ok is True
assert read_report.ok is True

assert loaded.metadata["dataset_id"] == trips_original.metadata["dataset_id"]
assert loaded.metadata["artifact_id"] == trips_original.metadata["artifact_id"]
assert loaded.metadata["is_validated"] is False

assert_loaded_data_equivalent(loaded.data, data_original)

assert loaded.schema.to_dict() == schema_original.to_dict()
assert loaded.schema_effective.to_dict() == schema_effective_original.to_dict()

ops_loaded = [ev["op"] for ev in loaded.metadata["events"]]
assert "write_trips" in ops_loaded
assert ops_loaded[-1] == "read_trips"

assert_issue_present(read_report.issues, "READ.METADATA.VALIDATED_FORCED_FALSE")

print("ops_loaded =", ops_loaded)
display(write_report.summary)
display(read_report.summary)
show_ok("Test 3.11 - round-trip mínimo write/read")

ops_loaded = ['write_trips', 'read_trips']


{'n_rows': 3,
 'files_written': ['trips.parquet', 'trips.metadata.json'],
 'path': 'tmp_op07_read_trips_smoke\\case_11_roundtrip_minimal\\artifact',
 'dataset_id': 'dset_test_001',
 'artifact_id': 'art_a966e570-e0ba-4745-937c-cf53d59cbe49',
 'dataset_id_status': 'preserved',
 'storage_format': 'parquet'}

{'n_rows': 3,
 'n_columns': 12,
 'path': 'tmp_op07_read_trips_smoke\\case_11_roundtrip_minimal\\artifact',
 'storage_format': 'parquet',
 'schema_source': 'metadata',
 'schema_mismatch': False,
 'dataset_id': 'dset_test_001',
 'dataset_id_status': 'loaded',
 'artifact_id': 'art_a966e570-e0ba-4745-937c-cf53d59cbe49',
 'artifact_id_status': 'loaded'}

OK - Test 3.11 - round-trip mínimo write/read
